# Практическое занятие 2. Работа с LLM через API

Ноутбук подключается к GigaChat через LangChain, проверяет связь и извлекает
количество проживающих из текстов заявок на аренду жилья.

Ключ доступа хранится в файле `.env` в переменной `GIGA_KEY` и не попадает
в репозиторий (`.env` добавлен в `.gitignore`).

## 1. Подключение к GigaChat

In [1]:
import os

from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat

load_dotenv()
giga_key = os.getenv("GIGA_KEY")

if not giga_key:
    raise ValueError("Ключ GIGA_KEY не найден в .env")

llm = GigaChat(
    credentials=giga_key,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

print(llm.invoke("Привет! Как дела?").content)

Привет! Всё отлично, спасибо. Готов обсудить что-нибудь интересное или помочь разобраться с задачей. Чем займёмся?


## 2. Базовый промпт и цепочка LangChain

Промпт формулирует задачу, модель её выполняет, парсер `StrOutputParser`
возвращает результат как строку.

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:""",
)

chain = basic_prompt | llm | StrOutputParser()

test_texts = [
    "Ищу квартиру для семьи из четырех человек",
    "Нужна студия для одного",
    "Семья с двумя детьми ищет жилье",
]

for text in test_texts:
    print(f"Текст: {text}")
    print(f"Результат: {chain.invoke({'text': text})}")
    print("---")

Текст: Ищу квартиру для семьи из четырех человек


Результат: 4
---
Текст: Нужна студия для одного


Результат: 1


---
Текст: Семья с двумя детьми ищет жилье


Результат: 4
---


## 3. Индивидуальная работа: мои 15 заявок

Заявки берутся из файла `rental_17.csv`, который соответствует моему варианту.

In [3]:
import pandas as pd

df = pd.read_csv("rental_17.csv", sep=";")
test_texts = df["text"].tolist()
print("Заявок в наборе:", len(test_texts))

for text in test_texts:
    print(f"Текст: {text}")
    print(f"Результат: {chain.invoke({'text': text})}")
    print("---")

Заявок в наборе: 15
Текст: Сниму недорогое жилье в лазаревском на 6 чел 3 взр и 3 дет 30.07-09.08
Результат: 6
---
Текст: Добрый вечер! Ищем жилье на 2 человек, с 3 сентября на 7 дней.


Результат: 2
---
Текст: Ищем жильё на 4 взрослых с 31.08.18 по 07.08.18, желательно центр
Результат: 4
---
Текст: Ищем жильё 2 взрослых и ребёнок 6 лет. С 8 июля по 21 июля


Результат: 3
---
Текст: Здравствуйте, интересует жилье с 17.08 по 25.08 двое взрослых и девочка 14 лет.
Результат: 3
---
Текст: Интересует жилье в Сочи у пляжа "Ривьера" (максимально близко к морю!!!) с 8 по 15 июля. 1 номер/квартира/комната, 3 человека. Предложения в л/с


Результат: 3
---
Текст: Добрый вечер. Ищем жилье с 17.08 на 10 дней. Желательно с бассейном. 2е взрослых и ребенок 5 лет
Результат: 3
---
Текст: Здравствуйте, ищем жилье с 10.08по 20.08,два взрослых,два ребенка,не далеко от моря.


Результат: 4
---
Текст: Ищем жильё в Лазаревское заезд 13 июля на 10 дней, 2 взр, дети 2 года и 5 лет, предпочтения душ, туалет, холодильник, кондиционер в номере, парковка для авто,  рассмотрим варианты, не курим
Результат: 4
---
Текст: Ищем жилье эконом с 6 по 16 сентября для женщины с ребенком 6 лет недалеко от моря.Пишите в личку


Результат: 2
---
Текст: Сниму жилье в июне у моря 2 взрослых до 1500 в сутки
Результат: 2
---
Текст: Добрый день интересует жилье семья 5 человек двое взрослых и трое детей с 20 по 30 сентября на берегу моря цена пожайлуста в личку


Результат: 5
---
Текст: Здравствуйте, интересует жилье п.Лазаревское 2 взр + 2 реб. 6 и 8 лет с 08.08 по 21.08? Недорого! Рядом с морем!
Результат: 4
---
Текст: А море то далеко? Или есть какие то тайные тропы? 
Если я сниму жильё на этой улице , сколько мне пешком идти до моря ? А главное - сколько мне пешком идти обратно ? В гору. По жаре .


Результат: Не удалось извлечь конкретное количество человек из текста заявки.

По контексту определить точное число нельзя.
---
Текст: Ищем не дорогое жильё на начало июля , не далеко от моря ..,на 2х
Результат: 2
---
